# SmolVLM-500M-Instruct — accuracy-max recipe (DoRA + captions + ensembling)


## 1. Install dependencies

In [1]:
# Pin versions known to play nicely with SmolVLM (Idefics3) + PEFT-DoRA.
# We also uninstall torchao because Colab's stock 0.10.0 trips PEFT's strict
# version check; we don't use any torchao quantization in this notebook.
!pip install -q --upgrade \
    "transformers>=4.46.0,<4.50" \
    "peft>=0.13.0" \
    "accelerate>=1.0.0" \
    "pillow==10.3.0" \
    "pandas" \
    "tqdm"
!pip uninstall -y -q torchao
print("\n✓ Installed. Now: Runtime → Restart session, then continue from cell 2.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 135.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 133.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 106.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incom

## 2. Mount Drive + imports + A100 runtime knobs

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
from google.colab import drive
drive.mount('/content/drive')

print("Content:")
!ls /content

print("\nOutputs:")
!ls /content/outputs || echo "no outputs folder"

print("\nDrive files:")
!ls /content/drive/MyDrive | head

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Content:
drive  images.zip  sample_data	test.csv  train.csv  val.csv

Outputs:
ls: cannot access '/content/outputs': No such file or directory
no outputs folder

Drive files:
AI FINAL PROJECT.gdoc
Assignment 1 AI.ipynb
Classroom
Colab Notebooks
CS-GY 6063: Software Engineering
CS-GY 6613: AI
Deep Learning HW1 Spring 26 P4,5,6 Starter.ipynb
demo13-diffusion.ipynb
dl-starter-notebook-v2.ipynb
Final_Basketball_AI_How_To_Detect_Track_and_Identify_Basketball_Players.ipynb


In [4]:
import os, gc, json, math, random, ast, re, time, pickle
from pathlib import Path
from typing import List, Dict, Any, Optional

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

import torchvision.transforms as T
from transformers import AutoProcessor, AutoModelForVision2Seq, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model

# Verify torchao is gone
try:
    import torchao
    print(f"!! torchao still present (v{torchao.__version__}) — re-run install + restart")
except ImportError:
    print("✓ torchao not present, PEFT will skip its dispatcher")

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0); cc = torch.cuda.get_device_capability(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"device: {name}  (sm_{cc[0]}{cc[1]}, {total_mem:.1f} GB)")


✓ torchao not present, PEFT will skip its dispatcher
torch: 2.10.0+cu128 | cuda: True
device: NVIDIA A100-SXM4-40GB  (sm_80, 42.4 GB)


In [5]:
# A100 throughput knobs
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

def gpu_mem_summary(tag=""):
    if not torch.cuda.is_available(): return
    used = torch.cuda.memory_allocated() / 1e9
    peak = torch.cuda.max_memory_allocated() / 1e9
    print(f"[mem{(' ' + tag) if tag else ''}] used={used:.2f}GB peak={peak:.2f}GB")

def reset_peak():
    torch.cuda.reset_peak_memory_stats()


## 3. Configuration

In [7]:
!unzip -q /content/images.zip -d /content/

In [9]:
!ls /content/images | head

test
train
val


In [11]:
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────
DATA_ROOT = Path("/content")

TRAIN_CSV = DATA_ROOT / "train.csv"
VAL_CSV   = DATA_ROOT / "val.csv"
TEST_CSV  = DATA_ROOT / "test.csv"

IMG_ROOT  = DATA_ROOT / "images"

# SAVE EVERYTHING TO DRIVE
OUTPUT_DIR = Path("/content/drive/MyDrive/endterm-kagcomp-outputs")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# ── Model ────────────────────────────────────────────────────────────────
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
ATTENTION_IMPL = "sdpa"

# ── Seed ────────────────────────────────────────────────────────────────
SEED = 42

# ── Training hyperparams ─────────────────────────────────────────────────
EPOCHS = 6

# safer but stronger-than-before setup
BATCH_SIZE = 2
GRAD_ACCUM = 8   # effective batch = 16

LR = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.08
MAX_GRAD_NORM = 1.0

LOG_EVERY = 25
EVAL_EVERY_FRAC = 0.34

USE_GRAD_CHECKPOINT = True

# ── LoRA / DoRA ──────────────────────────────────────────────────────────
LORA_RANK = 10
LORA_ALPHA = 20
LORA_DROPOUT = 0.05
USE_DORA = True

TARGET_MODULES = [
    "q_proj",
    "v_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

# ── Image processing ─────────────────────────────────────────────────────
# slightly stronger than your stable 1024 run
DO_IMAGE_SPLITTING = False
LONGEST_EDGE = 1280

# ── Inference ────────────────────────────────────────────────────────────
INFER_BATCH = 4
TTA_PASSES = 5
USE_CLOZE = False

# ── Captions ─────────────────────────────────────────────────────────────
USE_CAPTIONS = True
CAPTIONS_PATH = OUTPUT_DIR / "captions_qcond.json"
QCOND_CAPTIONS = True

# ── Prompt augmentation ──────────────────────────────────────────────────
PROMPT_VARIANTS = [
    {
        "answer_phrase": "Answer:",
        "choices_header": "Choices",
    },
    {
        "answer_phrase": "The correct answer is:",
        "choices_header": "Choices",
    },
    {
        "answer_phrase": "Therefore the correct option letter is:",
        "choices_header": "Options",
    },
]

# ── DataLoader ───────────────────────────────────────────────────────────
NUM_WORKERS_TRAIN = 4
NUM_WORKERS_INFER = 0
PIN_MEMORY = True

# ── Misc ─────────────────────────────────────────────────────────────────
DTYPE = torch.bfloat16
DEVICE = "cuda"

BEST_CKPT_PATH = OUTPUT_DIR / f"best_adapter_seed{SEED}"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"✓ Config loaded. Seed={SEED}")
print(f"✓ Output dir: {OUTPUT_DIR}")
print(f"✓ LONGEST_EDGE={LONGEST_EDGE}")
print(f"✓ TTA_PASSES={TTA_PASSES}")

✓ Config loaded. Seed=42
✓ Output dir: /content/drive/MyDrive/endterm-kagcomp-outputs
✓ LONGEST_EDGE=1280
✓ TTA_PASSES=5


## 4. Load + parse data

In [12]:
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)
print("train:", train_df.shape, "| val:", val_df.shape, "| test:", test_df.shape)
print("columns:", train_df.columns.tolist())


train: (3109, 15) | val: (1048, 15) | test: (1008, 13)
columns: ['id', 'image_path', 'question', 'choices', 'num_choices', 'answer', 'hint', 'lecture', 'solution', 'task', 'grade', 'subject', 'topic', 'category', 'skill']


In [14]:
def pick(df, *candidates, required=True):
    for c in candidates:
        if c in df.columns: return c
    if required: raise KeyError(f"None of {candidates} in {list(df.columns)}")
    return None

ID_COL       = pick(train_df, "id", "Id", "ID")
QUESTION_COL = pick(train_df, "question", "Question", "prompt")
CHOICES_COL  = pick(train_df, "choices", "options", "Choices", "Options")
ANSWER_COL   = pick(train_df, "answer", "label", "Answer", "correct_answer")
IMAGE_COL    = pick(train_df, "image", "image_path", "filename", "img", "image_name", required=False)
SUBJECT_COL  = pick(train_df, "subject", "Subject", required=False)
GRADE_COL    = pick(train_df, "grade", "Grade", "level", required=False)
TOPIC_COL    = pick(train_df, "topic", "Topic", "category", required=False)

print(dict(id=ID_COL, q=QUESTION_COL, choices=CHOICES_COL, ans=ANSWER_COL,
           img=IMAGE_COL, subj=SUBJECT_COL, grade=GRADE_COL, topic=TOPIC_COL))


{'id': 'id', 'q': 'question', 'choices': 'choices', 'ans': 'answer', 'img': 'image_path', 'subj': 'subject', 'grade': 'grade', 'topic': 'topic'}


In [15]:
def parse_choices(c):
    if isinstance(c, list): return [str(x) for x in c]
    if pd.isna(c): return []
    s = str(c).strip()
    if s.startswith('[') and s.endswith(']'):
        for parser in (json.loads, ast.literal_eval):
            try:
                v = parser(s)
                if isinstance(v, (list, tuple)): return [str(x) for x in v]
            except Exception: pass
    for sep in ['||', '\n', '\t', ' | ', '|', ';;']:
        if sep in s: return [p.strip() for p in s.split(sep) if p.strip()]
    return [s]

for df in (train_df, val_df, test_df):
    df['_choices'] = df[CHOICES_COL].apply(parse_choices)

print("Choice-count distribution (train):")
print(train_df['_choices'].apply(len).value_counts().sort_index())


Choice-count distribution (train):
_choices
2     664
3    1552
4     783
5     110
Name: count, dtype: int64


In [16]:
def resolve_image_path(row) -> Path:
    rid = str(row[ID_COL])
    split = rid.split('_', 1)[0] if '_' in rid else None
    cands = []
    if IMAGE_COL and isinstance(row.get(IMAGE_COL), str) and row[IMAGE_COL]:
        raw = row[IMAGE_COL].lstrip('/')
        stripped = re.sub(r'^(images/)+', '', raw)
        base = Path(raw).name
        cands += [IMG_ROOT / raw, IMG_ROOT / stripped,
                  (IMG_ROOT / split / base) if split else None,
                  IMG_ROOT / base]
    splits_to_try = ([split] if split else []) + ['train', 'val', 'test', '']
    for sp in splits_to_try:
        for ext in ['.png', '.jpg', '.jpeg', '.webp']:
            cands.append((IMG_ROOT / sp / f"{rid}{ext}") if sp else (IMG_ROOT / f"{rid}{ext}"))
    for c in cands:
        if c is not None and c.exists(): return c
    raise FileNotFoundError(f"id={rid}: no image found under {IMG_ROOT}")

# Sanity check
for df, name in [(train_df,'train'),(val_df,'val'),(test_df,'test')]:
    p = resolve_image_path(df.iloc[0])
    print(f"{name}: {p}  ({Image.open(p).size})")


train: /content/images/train/train_07667.png  ((302, 252))
val: /content/images/val/val_00671.png  ((302, 202))
test: /content/images/test/test_01750.png  ((302, 202))


## 5. Processor + base model

In [17]:
processor = AutoProcessor.from_pretrained(
    MODEL_ID, do_image_splitting=DO_IMAGE_SPLITTING,
    size={"longest_edge": LONGEST_EDGE},
)
tokenizer = processor.tokenizer
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Image processor size:", processor.image_processor.size)


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Image processor size: {'longest_edge': 1280}


In [18]:
def load_model(attn_impl: str):
    """SmolVLM = Idefics3. Vision tower lacks SDPA in older transformers,
    so set per-submodel: fast on the LM, eager on the small vision tower."""
    attempts = []
    if attn_impl != "eager":
        attempts.append({"text_config": attn_impl, "vision_config": "eager"})
        attempts.append({"text_model": attn_impl, "vision_model": "eager"})
    attempts.append(attn_impl)
    attempts.append("eager")
    last_err = None
    for impl in attempts:
        try:
            m = AutoModelForVision2Seq.from_pretrained(
                MODEL_ID, torch_dtype=DTYPE, _attn_implementation=impl,
            ).to(DEVICE)
            print(f"✓ Loaded with attention impl: {impl}")
            return m
        except (ImportError, ValueError, RuntimeError, TypeError, KeyError) as e:
            last_err = e
            print(f"  attempt impl={impl!r} failed: {type(e).__name__}: {e}")
    raise last_err

reset_peak()
base_model = load_model(ATTENTION_IMPL)   # we keep a reference for caption pass
base_model.config.use_cache = False
gpu_mem_summary("after base load")
print(f"Base params: {sum(p.numel() for p in base_model.parameters())/1e6:.1f}M")


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

✓ Loaded with attention impl: {'text_config': 'sdpa', 'vision_config': 'eager'}
[mem after base load] used=1.04GB peak=1.04GB
Base params: 507.5M


## 6. Prompt construction

Single source-of-truth function used by training, captioning, and all 3 inference paths.

In [20]:
LETTERS = "ABCDEFGHIJ"

def build_user_prompt(row, choices, *, caption=None,
                      answer_phrase="Answer:", choices_header="Choices"):
    parts = []
    meta = []
    for col, label in [(SUBJECT_COL,"Subject"),(TOPIC_COL,"Topic"),(GRADE_COL,"Grade")]:
        if col and pd.notna(row.get(col)): meta.append(f"{label}: {row[col]}")
    if meta: parts.append(" | ".join(meta))
    if caption: parts.append(f"Image description: {caption}")
    parts.append(f"Question: {row[QUESTION_COL]}")
    parts.append(f"{choices_header}:\n" + "\n".join(f"{LETTERS[i]}. {c}" for i,c in enumerate(choices)))
    parts.append(answer_phrase)
    return "\n\n".join(parts)


def messages_for_row(row, choices, *, caption=None, with_answer=None,
                     answer_phrase="Answer:", choices_header="Choices"):
    user_text = build_user_prompt(row, choices, caption=caption,
                                  answer_phrase=answer_phrase, choices_header=choices_header)
    msgs = [{"role":"user","content":[{"type":"image"},{"type":"text","text":user_text}]}]
    if with_answer is not None:
        msgs.append({"role":"assistant","content":[{"type":"text","text":LETTERS[with_answer]}]})
    return msgs


# Sanity check
ex = train_df.iloc[0]
print(build_user_prompt(ex, ex['_choices']))
print("\nGold:", LETTERS[int(ex[ANSWER_COL])])


Subject: natural science | Topic: literacy-in-science | Grade: grade8

Question: Why might putting each tadpole in its own pool of water increase the reproductive success of a male Amazonian poison frog? Complete the claim below that answers this question and is best supported by the passage.
Putting each tadpole in its own pool of water increases the chances that ().

Choices:
A. the male's tadpoles will be larger when they hatch
B. the male will carry his tadpoles through the forest
C. the male's tadpoles will become adult frogs

Answer:

Gold: C


## 7. Image augmentation (train + TTA)

In [21]:
class TrainImageAug:
    def __init__(self):
        self.color  = T.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.05)
        self.affine = T.RandomAffine(degrees=2, translate=(0.02,0.02), scale=(0.97,1.03), fill=255)
    def __call__(self, img):
        if random.random() < 0.5: img = self.color(img)
        if random.random() < 0.3: img = self.affine(img)
        return img

train_aug = TrainImageAug()

def load_image(path: Path, train: bool = False) -> Image.Image:
    img = Image.open(path).convert("RGB")
    img = ImageOps.exif_transpose(img)
    if train: img = train_aug(img)
    return img


## 8. Question-conditioned caption pass

**Run this BEFORE applying LoRA.** The captions go into a JSON cache keyed by `id`. If the cache exists from a prior run, this cell just loads it. Generic captioning gives generic descriptions; question-conditioned captioning produces descriptions of *the parts of the image relevant to this question*, which composes much better with the model's MCQ reasoning.

In [22]:
@torch.no_grad()
def generate_captions(df, out_path: Path, batch=8, qcond=True):
    if out_path.exists():
        with open(out_path) as f: cache = json.load(f)
    else:
        cache = {}
    base_model.eval()

    todo = [r for _, r in df.iterrows() if str(r[ID_COL]) not in cache]
    if not todo:
        print(f"All {len(cache)} captions already cached at {out_path}")
        return cache

    pbar = tqdm(range(0, len(todo), batch), desc="captioning")
    for start in pbar:
        chunk = todo[start:start+batch]
        images, texts, rids = [], [], []
        for row in chunk:
            try:
                img = load_image(resolve_image_path(row), train=False)
            except Exception:
                cache[str(row[ID_COL])] = ""; continue
            if qcond:
                q = str(row[QUESTION_COL]).strip()
                cap_prompt = (f"Looking at this image, describe in 1-2 factual sentences "
                              f"the visual elements (text labels, axes, numbers, objects, "
                              f"colors, shapes) most relevant to answering: '{q}'")
            else:
                cap_prompt = ("Describe this image in one or two factual sentences. "
                              "Focus on text labels, axes, numbers, and key objects.")
            msgs = [{"role":"user","content":[{"type":"image"},{"type":"text","text":cap_prompt}]}]
            texts.append(processor.apply_chat_template(msgs, add_generation_prompt=True))
            images.append([img])
            rids.append(str(row[ID_COL]))
        if not rids: continue

        inputs = processor(text=texts, images=images, return_tensors="pt", padding=True)
        pv = inputs.pop('pixel_values').to(DEVICE, dtype=DTYPE)
        inputs = {k: v.to(DEVICE) for k,v in inputs.items()}
        inputs['pixel_values'] = pv
        out = base_model.generate(**inputs, max_new_tokens=80, do_sample=False, use_cache=True)
        gen = out[:, inputs['input_ids'].shape[1]:]
        decoded = processor.batch_decode(gen, skip_special_tokens=True)
        for rid, cap in zip(rids, decoded):
            cache[rid] = cap.strip()

        if start % (batch * 25) == 0:
            with open(out_path, "w") as f: json.dump(cache, f)

    with open(out_path, "w") as f: json.dump(cache, f)
    print(f"Saved {len(cache)} captions → {out_path}")
    return cache


if USE_CAPTIONS:
    all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    CAPTIONS = generate_captions(all_df, CAPTIONS_PATH, batch=4, qcond=QCOND_CAPTIONS)
    # Sanity-check a few
    for rid in list(CAPTIONS.keys())[:3]:
        print(f"\n[{rid}]", CAPTIONS[rid][:200])
else:
    CAPTIONS = {}
    print("USE_CAPTIONS=False — skipping caption pass")


captioning:   0%|          | 0/1292 [00:00<?, ?it/s]

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='le

Saved 5165 captions → /content/drive/MyDrive/endterm-kagcomp-outputs/captions_qcond.json

[train_07667] Putting each tadpole in its own pool of water increases the chances that ().

[train_02628] at




at
at


 rh

[train_00927] intelligently designed and optimized for the specific needs of the business.  The business is a small, family-owned company that specializes in the production of high-quality, sustainable products.  T


## 9. Apply DoRA

Param math: 5 modules × `r=10` × DoRA + magnitude vectors over 32 layers ≈ **4.64M** trainable. Asserts before continuing.

In [23]:
lora_config = LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias="none", use_dora=USE_DORA,
    target_modules=TARGET_MODULES,
    task_type="CAUSAL_LM",
    exclude_modules=r".*vision_model.*",   # vision tower stays frozen
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable: {trainable:,} ({trainable/1e6:.3f}M)")
assert trainable <= 5_000_000, f"Over 5M cap! ({trainable:,})"
print(f"Headroom: {5_000_000 - trainable:,}")

if USE_GRAD_CHECKPOINT:
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    print("Gradient checkpointing: ON")


trainable params: 4,638,720 || all params: 512,121,024 || trainable%: 0.9058

Trainable: 4,638,720 (4.639M)
Headroom: 361,280
Gradient checkpointing: ON


## 10. Dataset + CPU-only collators with prompt-variant sampling

Collators run inside DataLoader workers. They must NOT touch CUDA — that's why we keep `.to(DEVICE)` outside in the training loop. During training we randomly pick one of `PROMPT_VARIANTS` per example; during inference we use the canonical "Answer:" form (the multi-prompt cell handles variants explicitly).

In [24]:
class MCQDataset(Dataset):
    def __init__(self, df, train=False):
        self.df = df.reset_index(drop=True)
        self.train = train
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        rid = str(row[ID_COL])
        return {
            "id": rid,
            "image": load_image(resolve_image_path(row), train=self.train),
            "row": row.to_dict(),
            "choices": row['_choices'],
            "caption": CAPTIONS.get(rid) if USE_CAPTIONS else None,
            "answer_idx": int(row[ANSWER_COL]) if ANSWER_COL in row and pd.notna(row[ANSWER_COL]) else None,
            "n_choices": len(row['_choices']),
        }


def collate_train(batch):
    """Random prompt variant per example. CPU only."""
    texts, prompt_only_texts, images = [], [], []
    for ex in batch:
        v = random.choice(PROMPT_VARIANTS)
        msgs_full = messages_for_row(ex['row'], ex['choices'], caption=ex['caption'],
                                     with_answer=ex['answer_idx'], **v)
        msgs_prompt = messages_for_row(ex['row'], ex['choices'], caption=ex['caption'], **v)
        texts.append(processor.apply_chat_template(msgs_full, add_generation_prompt=False))
        prompt_only_texts.append(processor.apply_chat_template(msgs_prompt, add_generation_prompt=True))
        images.append([ex['image']])

    inputs = processor(text=texts, images=images, return_tensors="pt", padding=True)
    prompt_only_ids = processor(text=prompt_only_texts, images=images,
                                return_tensors="pt", padding=True)['input_ids']

    pad_id = tokenizer.pad_token_id
    labels = inputs['input_ids'].clone()
    for i in range(labels.size(0)):
        prompt_len = (prompt_only_ids[i] != pad_id).sum().item()
        labels[i, :prompt_len] = -100
    labels[inputs['input_ids'] == pad_id] = -100

    out = dict(inputs)
    out['pixel_values'] = out['pixel_values'].to(dtype=DTYPE)
    out['labels'] = labels
    return out


def collate_eval(batch):
    """Canonical 'Answer:' prompt for vanilla eval. Multi-prompt is separate."""
    texts, images, metas = [], [], []
    for ex in batch:
        msgs = messages_for_row(ex['row'], ex['choices'], caption=ex['caption'])
        texts.append(processor.apply_chat_template(msgs, add_generation_prompt=True))
        images.append([ex['image']])
        metas.append({"id":ex['id'],"n_choices":ex['n_choices'],"answer_idx":ex['answer_idx']})

    inputs = processor(text=texts, images=images, return_tensors="pt", padding=True)
    out = dict(inputs)
    out['pixel_values'] = out['pixel_values'].to(dtype=DTYPE)
    return out, metas


def move_to_device(batch):
    return {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}


## 11. Letter token IDs

In [25]:
def find_letter_token_ids(n_letters=10):
    """Verify letter tokenization is consistent across all prompt variants."""
    dummy_row = train_df.iloc[0]
    sets = []
    for v in PROMPT_VARIANTS + [{}]:
        msgs = messages_for_row(dummy_row, dummy_row['_choices'], **v)
        prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
        prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
        ids = []
        for L in LETTERS[:n_letters]:
            full_ids = tokenizer(prompt + L, add_special_tokens=False)['input_ids']
            if len(full_ids) <= len(prompt_ids):
                raise RuntimeError(f"No new token for '{L}' in variant {v}")
            ids.append(full_ids[len(prompt_ids)])
        sets.append(ids)
    # All variants should give identical letter token IDs (single-char letters
    # tokenize the same way regardless of preceding context for SmolLM2 BPE)
    for s in sets[1:]:
        if s != sets[0]:
            print("⚠ Letter token IDs differ between prompt variants!")
            print("  This means each variant needs its own LETTER_TOKEN_IDS.")
            return sets
    print(f"✓ Letter tokens consistent across all variants: {dict(zip(LETTERS[:n_letters], sets[0]))}")
    return sets[0]

LETTER_TOKEN_IDS = find_letter_token_ids(10)
if isinstance(LETTER_TOKEN_IDS[0], list):
    raise RuntimeError("Letter tokens differ across variants — need to refactor scoring.")
assert len(set(LETTER_TOKEN_IDS)) == len(LETTER_TOKEN_IDS)


✓ Letter tokens consistent across all variants: {'A': 49, 'B': 50, 'C': 51, 'D': 52, 'E': 53, 'F': 54, 'G': 55, 'H': 56, 'I': 57, 'J': 58}


## 12. DataLoaders + optimizer

In [26]:
train_ds = MCQDataset(train_df, train=True)
val_ds   = MCQDataset(val_df,   train=False)
test_ds  = MCQDataset(test_df,  train=False)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_train,
    drop_last=True, num_workers=NUM_WORKERS_TRAIN, pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS_TRAIN > 0,
    prefetch_factor=2 if NUM_WORKERS_TRAIN > 0 else None,
)
val_loader   = DataLoader(val_ds,   batch_size=INFER_BATCH, shuffle=False,
                          collate_fn=collate_eval, num_workers=NUM_WORKERS_INFER, pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_ds,  batch_size=INFER_BATCH, shuffle=False,
                          collate_fn=collate_eval, num_workers=NUM_WORKERS_INFER, pin_memory=PIN_MEMORY)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps     = steps_per_epoch * EPOCHS
warmup_steps    = max(1, int(WARMUP_RATIO * total_steps))

optim = AdamW([p for p in model.parameters() if p.requires_grad],
              lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.999), fused=True)
scheduler = get_cosine_schedule_with_warmup(optim, warmup_steps, total_steps)

print(f"steps/epoch: {steps_per_epoch}, total: {total_steps}, warmup: {warmup_steps}")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}")


steps/epoch: 195, total: 1170, warmup: 93
Effective batch: 16


## 13. Validation (next-token logit scoring)

In [27]:
@torch.no_grad()
def evaluate_logits(loader, desc="val"):
    """Returns per-example raw logits over letter tokens. Caller decides what to do."""
    model.eval()
    rows = []
    for batch, metas in tqdm(loader, desc=desc, leave=False):
        batch = move_to_device(batch)
        out = model(**batch)
        attn = batch.get('attention_mask')
        last_pos = attn.sum(dim=1) - 1 if attn is not None else \
                   torch.full((batch['input_ids'].size(0),),
                              batch['input_ids'].size(1)-1, device=DEVICE)
        nl = out.logits[torch.arange(out.logits.size(0), device=DEVICE), last_pos]
        for i, m in enumerate(metas):
            n = m['n_choices']
            scores = nl[i, LETTER_TOKEN_IDS[:n]].float().cpu()
            rows.append({"id": m['id'], "n": n, "gold": m['answer_idx'], "scores": scores})
    model.train()
    return rows


def acc_from_rows(rows, bias=None):
    correct, total = 0, 0
    for r in rows:
        if r['gold'] is None: continue
        s = r['scores'] - (bias[:r['n']] if bias is not None else 0.0)
        if int(s.argmax().item()) == r['gold']:
            correct += 1
        total += 1
    return correct / max(1, total)


## 14. Training loop

In [28]:
best_val_acc = -1.0
global_step = 0
eval_every = max(1, int(steps_per_epoch * EVAL_EVERY_FRAC))

reset_peak()
model.train()
t0 = time.time()
val_history = []

for epoch in range(EPOCHS):
    pbar = tqdm(train_loader, desc=f"epoch {epoch+1}/{EPOCHS}")
    optim.zero_grad(set_to_none=True)
    running_loss, running_n = 0.0, 0

    for step, batch in enumerate(pbar):
        batch = move_to_device(batch)
        out = model(**batch)
        loss = out.loss / GRAD_ACCUM
        loss.backward()
        running_loss += out.loss.item(); running_n += 1

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], MAX_GRAD_NORM)
            optim.step()
            scheduler.step()
            optim.zero_grad(set_to_none=True)
            global_step += 1

            if global_step % LOG_EVERY == 0:
                pbar.set_postfix(loss=f"{running_loss/max(1,running_n):.4f}",
                                 lr=f"{scheduler.get_last_lr()[0]:.2e}",
                                 step=global_step)
                running_loss, running_n = 0.0, 0

            if global_step % eval_every == 0:
                rows = evaluate_logits(val_loader, desc=f"val@{global_step}")
                acc = acc_from_rows(rows)
                val_history.append((global_step, acc))
                print(f"\n[step {global_step}] val acc = {acc:.4f}  "
                      f"(elapsed {(time.time()-t0)/60:.1f} min)")
                gpu_mem_summary(f"step {global_step}")
                if acc > best_val_acc:
                    best_val_acc = acc
                    model.save_pretrained(BEST_CKPT_PATH)
                    print(f"  ↳ best, saved → {BEST_CKPT_PATH}")

    rows = evaluate_logits(val_loader, desc=f"val@epoch{epoch+1}")
    acc = acc_from_rows(rows)
    val_history.append((global_step, acc))
    print(f"[epoch {epoch+1}] val acc = {acc:.4f}  (elapsed {(time.time()-t0)/60:.1f} min)")
    if acc > best_val_acc:
        best_val_acc = acc
        model.save_pretrained(BEST_CKPT_PATH)
        print(f"  ↳ best, saved → {BEST_CKPT_PATH}")

print(f"\nBest val acc: {best_val_acc:.4f}")
print(f"Total training time: {(time.time()-t0)/60:.1f} min")
print(f"Adapter at: {BEST_CKPT_PATH}")

# Save val history for later analysis
with open(BEST_CKPT_PATH / "val_history.json", "w") as f:
    json.dump(val_history, f)


epoch 1/6:   0%|          | 0/1554 [00:00<?, ?it/s]

val@66:   0%|          | 0/262 [00:00<?, ?it/s]


[step 66] val acc = 0.5687  (elapsed 6.7 min)
[mem step 66] used=1.22GB peak=1.91GB
  ↳ best, saved → /content/drive/MyDrive/endterm-kagcomp-outputs/best_adapter_seed42


val@132:   0%|          | 0/262 [00:00<?, ?it/s]


[step 132] val acc = 0.6183  (elapsed 13.3 min)
[mem step 132] used=1.22GB peak=1.91GB
  ↳ best, saved → /content/drive/MyDrive/endterm-kagcomp-outputs/best_adapter_seed42


...


[step 1056] val acc = 0.7414  (elapsed 115.9 min)
[mem step 1056] used=1.20GB peak=1.92GB


val@1122:   0%|          | 0/262 [00:00<?, ?it/s]


[step 1122] val acc = 0.7500  (elapsed 122.6 min)
[mem step 1122] used=1.20GB peak=1.92GB
  ↳ best, saved → /content/drive/MyDrive/endterm-kagcomp-outputs/best_adapter_seed42


val@epoch6:   0%|          | 0/262 [00:00<?, ?it/s]

[epoch 6] val acc = 0.7462  (elapsed 127.5 min)

Best val acc: 0.7500
Total training time: 127.5 min
Adapter at: /content/drive/MyDrive/endterm-kagcomp-outputs/best_adapter_seed42


## 15. Reload best adapter

In [29]:
import safetensors.torch as st

adapter_file = BEST_CKPT_PATH / "adapter_model.safetensors"

if adapter_file.exists():
    state = st.load_file(str(adapter_file))
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"✓ Loaded best adapter. missing={len(missing)} unexpected={len(unexpected)}")
else:
    print("⚠ No best adapter file — using current weights.")

✓ Loaded best adapter. missing=969 unexpected=480


In [30]:
TTA_PASSES = 7
INFER_BATCH = 2

print("Final inference settings:")
print("TTA_PASSES:", TTA_PASSES)
print("INFER_BATCH:", INFER_BATCH)

Final inference settings:
TTA_PASSES: 7
INFER_BATCH: 2


## 16. Multi-prompt + TTA + letter-bias calibration

This is the heavyweight inference path. It composes three things:

1. **Multi-prompt scoring**: each example is scored under all `PROMPT_VARIANTS`, log-probs averaged.
2. **TTA**: each scoring pass is repeated `TTA_PASSES` times with image jitter; log-probs averaged.
3. **Letter-bias calibration**: per-letter mean log-prob estimated on val, subtracted at test time.

If calibration doesn't help on val, the code automatically skips it for test.

In [31]:
@torch.no_grad()
def score_with_variants_and_tta(ds, n_passes=5, batch_size=8, return_logprobs=False):
    """Returns one row per example with averaged log-probs over letters."""
    model.eval()
    aug = TrainImageAug()
    indices = list(range(len(ds)))
    rows = []

    for start in tqdm(range(0, len(indices), batch_size), desc="multi-prompt+TTA"):
        chunk = [ds[i] for i in indices[start:start+batch_size]]
        agg = [torch.zeros(ex['n_choices']) for ex in chunk]
        n_combinations = 0

        for variant in PROMPT_VARIANTS:
            for k in range(n_passes):
                texts, images = [], []
                for ex in chunk:
                    img = ex['image'] if k == 0 else aug(ex['image'])
                    msgs = messages_for_row(ex['row'], ex['choices'], caption=ex['caption'], **variant)
                    texts.append(processor.apply_chat_template(msgs, add_generation_prompt=True))
                    images.append([img])
                inputs = processor(text=texts, images=images, return_tensors="pt", padding=True)
                pv = inputs.pop('pixel_values').to(DEVICE, dtype=DTYPE)
                inputs = {kk: v.to(DEVICE) for kk, v in inputs.items()}
                inputs['pixel_values'] = pv
                out = model(**inputs)
                attn = inputs['attention_mask']
                last_pos = attn.sum(dim=1) - 1
                nl = out.logits[torch.arange(out.logits.size(0), device=DEVICE), last_pos]
                for i, ex in enumerate(chunk):
                    n = ex['n_choices']
                    lp = F.log_softmax(nl[i, LETTER_TOKEN_IDS[:n]], dim=-1).float().cpu()
                    agg[i] += lp
                n_combinations += 1

        for ex, a in zip(chunk, agg):
            a = a / n_combinations
            rows.append({
                "id": ex['id'], "n": ex['n_choices'],
                "gold": ex['answer_idx'], "logprobs": a,
            })

    return rows


In [32]:
# Estimate per-letter bias on val with the SAME inference path used at test.
# This makes calibration meaningful.
print("Scoring val with multi-prompt + TTA...")
val_rows = score_with_variants_and_tta(val_ds, n_passes=TTA_PASSES, batch_size=INFER_BATCH//2)

# Compute per-letter mean log-prob across val
max_n = max(r['n'] for r in val_rows)
bias = torch.zeros(max_n)
counts = torch.zeros(max_n)
for r in val_rows:
    bias[:r['n']] += r['logprobs']
    counts[:r['n']] += 1
bias = bias / counts.clamp(min=1)
print(f"Per-letter mean logprob (val): {[f'{x:+.3f}' for x in bias.tolist()]}")

# Decide whether to apply calibration based on val performance
def acc_from_logprob_rows(rows, b=None):
    correct, total = 0, 0
    for r in rows:
        if r['gold'] is None: continue
        s = r['logprobs'] - (b[:r['n']] if b is not None else 0.0)
        if int(s.argmax().item()) == r['gold']: correct += 1
        total += 1
    return correct / max(1, total)

val_raw = acc_from_logprob_rows(val_rows)
val_cal = acc_from_logprob_rows(val_rows, bias)
print(f"Val acc (multi-prompt + TTA, raw):        {val_raw:.4f}")
print(f"Val acc (multi-prompt + TTA, calibrated): {val_cal:.4f}")

APPLY_CALIBRATION = val_cal > val_raw + 0.001
print(f"\nWill {'apply' if APPLY_CALIBRATION else 'skip'} calibration on test.")


Scoring val with multi-prompt + TTA...


multi-prompt+TTA:   0%|          | 0/1048 [00:00<?, ?it/s]

Per-letter mean logprob (val): ['-4.960', '-4.117', '-4.653', '-6.839', '-1.529']
Val acc (multi-prompt + TTA, raw):        0.7405
Val acc (multi-prompt + TTA, calibrated): 0.7328

Will skip calibration on test.


In [33]:
# Score test with the same path, then decide on calibration
print("Scoring test with multi-prompt + TTA...")
test_rows = score_with_variants_and_tta(test_ds, n_passes=TTA_PASSES, batch_size=INFER_BATCH//2)

def predict_from_rows(rows, b=None):
    out = []
    for r in rows:
        s = r['logprobs'] - (b[:r['n']] if b is not None else 0.0)
        out.append({"id": r['id'], "answer": int(s.argmax().item()), "n": r['n'], "scores": s})
    return out

test_preds = predict_from_rows(test_rows, bias if APPLY_CALIBRATION else None)
print(f"✓ Predicted {len(test_preds)} test examples")

# Save ALL the scoring info — useful for ensembling later
with open(BEST_CKPT_PATH / "test_logprobs.pkl", "wb") as f:
    pickle.dump({"rows": test_rows, "bias": bias, "applied_calibration": APPLY_CALIBRATION}, f)


Scoring test with multi-prompt + TTA...


multi-prompt+TTA:   0%|          | 0/1008 [00:00<?, ?it/s]

✓ Predicted 1008 test examples


## 17. Write submission CSV

In [ ]:
sub = pd.DataFrame([{"id": p['id'], "answer": p['answer']} for p in test_preds])

sub = test_df[[ID_COL]].rename(columns={ID_COL: "id"}).merge(
    sub,
    on="id",
    how="left"
)

assert sub['answer'].notna().all(), "Some test ids missing predictions"

sub['answer'] = sub['answer'].astype(int)

sub_path = OUTPUT_DIR / "submission.csv"
sub.to_csv(sub_path, index=False)

# ALSO SAVE A CLEAR FINAL COPY
DRIVE_SUB_PATH = "/content/drive/MyDrive/final_submission.csv"
sub.to_csv(DRIVE_SUB_PATH, index=False)

print(f"✓ Wrote {sub_path}")
print(f"✓ Saved final submission to Drive: {DRIVE_SUB_PATH}")

print("\nPrediction distribution:")
print(sub['answer'].value_counts().sort_index())

sub.head()

In [35]:
DRIVE_SUB_PATH = "/content/drive/MyDrive/final_submission.csv"

sub.to_csv(DRIVE_SUB_PATH, index=False)

print(f"✓ Also saved submission to Drive: {DRIVE_SUB_PATH}")

✓ Also saved submission to Drive: /content/drive/MyDrive/final_submission.csv


In [36]:
from google.colab import files
files.download(str(sub_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 18. (Optional) Train+val joint finetune

Once you've confirmed val acc ≈ test acc on Kaggle's public leaderboard, you can squeeze out another ~0.5-1% by retraining on **train + val combined**. Only do this once at the very end — there's no validation signal during this phase, so use the same hyperparameters as your best train-only run and trust the schedule.

Skip this cell entirely if you're not at the very end of your iteration.

In [ ]:
DO_TRAINVAL_FINETUNE = False  # flip to True only for your final submission

if DO_TRAINVAL_FINETUNE:
    print("Combining train + val and retraining for 2 more epochs...")
    combined_df = pd.concat([train_df, val_df], ignore_index=True)
    combined_ds = MCQDataset(combined_df, train=True)
    combined_loader = DataLoader(
        combined_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_train,
        drop_last=True, num_workers=NUM_WORKERS_TRAIN, pin_memory=PIN_MEMORY,
        persistent_workers=NUM_WORKERS_TRAIN > 0,
        prefetch_factor=2 if NUM_WORKERS_TRAIN > 0 else None,
    )

    extra_epochs = 2
    extra_steps = math.ceil(len(combined_loader) / GRAD_ACCUM) * extra_epochs
    optim2 = AdamW([p for p in model.parameters() if p.requires_grad],
                   lr=LR/3, weight_decay=WEIGHT_DECAY, fused=True)
    scheduler2 = get_cosine_schedule_with_warmup(optim2, max(1, int(0.05*extra_steps)), extra_steps)

    model.train()
    optim2.zero_grad(set_to_none=True)
    for epoch in range(extra_epochs):
        for step, batch in enumerate(tqdm(combined_loader, desc=f"trainval ep{epoch+1}")):
            batch = move_to_device(batch)
            out = model(**batch)
            (out.loss / GRAD_ACCUM).backward()
            if (step+1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], MAX_GRAD_NORM)
                optim2.step(); scheduler2.step()
                optim2.zero_grad(set_to_none=True)

    final_path = OUTPUT_DIR / f"final_adapter_seed{SEED}"
    model.save_pretrained(final_path)
    print(f"✓ Saved trainval-finetuned adapter to {final_path}")
    # Re-run section 16+17 to regenerate submission.csv with this adapter.


## 19. (Optional) Multi-seed ensemble

To get the last ~1-2 pts: run this whole notebook 2 more times with `SEED=123` and `SEED=7`. Each run saves its predictions to `BEST_CKPT_PATH / "test_logprobs.pkl"`. Then run this cell once at the end to average all three.

In [ ]:
ENSEMBLE_SEEDS = [42, 123, 7]   # only files that exist will be used
ENSEMBLE_DIRS = [OUTPUT_DIR / f"best_adapter_seed{s}" for s in ENSEMBLE_SEEDS]

available = [d for d in ENSEMBLE_DIRS if (d / "test_logprobs.pkl").exists()]
print(f"Found {len(available)} of {len(ENSEMBLE_DIRS)} seed runs:")
for d in available: print(f"  {d}")

if len(available) >= 2:
    all_runs = []
    for d in available:
        with open(d / "test_logprobs.pkl", "rb") as f:
            all_runs.append(pickle.load(f))

    # Average log-probs across runs (per id, per letter)
    by_id = {}
    for run in all_runs:
        bias = run['bias'] if run['applied_calibration'] else None
        for r in run['rows']:
            s = r['logprobs'] - (bias[:r['n']] if bias is not None else 0.0)
            if r['id'] not in by_id:
                by_id[r['id']] = {"n": r['n'], "sum": torch.zeros(r['n']), "count": 0}
            by_id[r['id']]["sum"] += s
            by_id[r['id']]["count"] += 1

    ens_preds = []
    for rid, v in by_id.items():
        avg = v['sum'] / v['count']
        ens_preds.append({"id": rid, "answer": int(avg.argmax().item())})

    sub_ens = pd.DataFrame(ens_preds)
    sub_ens = test_df[[ID_COL]].rename(columns={ID_COL: "id"}).merge(sub_ens, on="id", how="left")
    sub_ens['answer'] = sub_ens['answer'].astype(int)
    sub_ens_path = OUTPUT_DIR / "submission_ensemble.csv"
    sub_ens.to_csv(sub_ens_path, index=False)
    print(f"\n✓ Wrote ensemble submission: {sub_ens_path}")
    print(sub_ens['answer'].value_counts().sort_index())
else:
    print("\nNeed at least 2 seed runs to ensemble. Re-run the notebook with SEED=123 and SEED=7 first.")
